# Alternative feature selection - 03 outer-fold evaluation

This notebook consumes only artifacts created by notebooks 01 and 02. It now selects global candidate `recipe_config_id` values from notebook-02 round-2 scores and audits each selected configuration on every outer fold.

The key invariant is that final `stable_outer_configurations.csv` may only contain configurations with `outer_fold_count == 5`. This avoids survivorship bias from evaluating a config only on the folds where it survived notebook-02 inner-OOF prefiltering.

For final selection this notebook uses a rate-matched and population-scaled outer score. The final submission selects top 1000 from the full test set, so each 1000-row outer fold is capped at about 200 rows. The TP/FP gross value from that fold is then scaled back to the full test population, while feature cost is charged once.


## 0. Setup


In [1]:
from __future__ import annotations

import json
import os
from datetime import UTC, datetime

import pandas as pd

from cost_effective.dataset import find_project_root, load_test_data, load_training_data
from cost_effective.dataset.utils import DEFAULT_MAX_TARGETS
from cost_effective.models.feature_selection_alternative_evaluation import (
    aggregate_outer_scores,
    complete_outer_prediction_pairs,
    filter_outer_recipe_pairs,
    rescore_outer_predictions_at_targets,
    run_outer_fold_evaluation,
    select_cost_aware_configurations,
    select_global_stage02_pipeline_recipes,
    select_stable_configurations,
)
from cost_effective.models.feature_selection_alternative_inner_selection import (
    load_stage_one_tables,
)
from cost_effective.notebook_artifacts import read_csv_if_available

project_root = find_project_root()
USE_EXISTING_OUTPUTS = True
FORCE_RERUN = False

stage_one_dir = (
    project_root / "outputs" / "feature_selection_alternative" / "01_fold_plan_and_recipe_space"
)
stage_two_dir = (
    project_root / "outputs" / "feature_selection_alternative" / "02_inner_recipe_selection"
)
outputs = project_root / "outputs" / "feature_selection_alternative" / "03_fold_evaluation"
outputs.mkdir(parents=True, exist_ok=True)

required_stage_one_files = (
    "outer_fold_assignments.csv",
    "inner_fold_assignments.csv",
    "prescreen_recipe_space.csv",
    "feature_size_grid.csv",
    "model_spec_space.csv",
    "pipeline_recipe_space.csv",
    "leakage_contract.csv",
)
missing_stage_one_files = [
    name for name in required_stage_one_files if not (stage_one_dir / name).exists()
]
if missing_stage_one_files:
    raise FileNotFoundError(
        "Missing setup artifacts for the alternative feature-selection path: "
        f"{missing_stage_one_files}. Run "
        "notebooks/alternative_approach/modeling_feature_selection_alternative_setup.ipynb first."
    )

required_stage_two_files = ("round2_inner_oof_scores.csv",)
missing_stage_two_files = [
    name for name in required_stage_two_files if not (stage_two_dir / name).exists()
]
if missing_stage_two_files:
    raise FileNotFoundError(
        "Missing inner-selection artifacts: "
        f"{missing_stage_two_files}. Run "
        "notebooks/alternative_approach/modeling_feature_selection_alternative_inner_selection.ipynb first."
    )

X_train, y_train = load_training_data(project_root / "data")
X_test_row_count = len(load_test_data(project_root / "data"))
stage_one = load_stage_one_tables(stage_one_dir)

outer_assignments = stage_one["outer_assignments"]
inner_assignments = stage_one["inner_assignments"]
prescreen_recipes = stage_one["prescreen_recipes"]
model_specs = stage_one["model_specs"]

X_train.shape, y_train.shape, outer_assignments.shape, X_test_row_count

((5000, 500), (5000,), (5000, 2), 5000)

In [2]:
RANDOM_STATE = 42
MAX_TARGETS = DEFAULT_MAX_TARGETS
OUTER_FOLDS_TO_RUN = None  # None means all outer folds declared in notebook 01.

STAGE03_GLOBAL_CONFIGS = int(os.environ.get("ALTERNATIVE_FS_EVAL_GLOBAL_CONFIGS", "20"))
STAGE03_MAX_FEATURE_SIZE = int(os.environ.get("ALTERNATIVE_FS_EVAL_MAX_FEATURE_SIZE", "6"))
MIN_INNER_FOLD_COUNT_ROUND2 = 5

RUN_OUTER_REFIT = bool(int(os.environ.get("ALTERNATIVE_FS_EVAL_RUN_REFIT", "1")))

TEST_TOP_N = DEFAULT_MAX_TARGETS
TEST_ROW_COUNT = X_test_row_count
TEST_CONTACT_RATE = min(1.0, float(TEST_TOP_N) / float(TEST_ROW_COUNT))
USE_RATE_MATCHED_OUTER_SELECTION = True
COST_AWARE_MAX_FEATURE_SIZE = STAGE03_MAX_FEATURE_SIZE
COST_AWARE_FALLBACK_MIN_OUTER_FOLDS = None
TOP_STABLE_CONFIGS = 10
MIN_OUTER_FOLDS_FOR_STABILITY = None

config = {
    "random_state": RANDOM_STATE,
    "max_targets": MAX_TARGETS,
    "outer_folds_to_run": OUTER_FOLDS_TO_RUN,
    "stage03_global_configs": STAGE03_GLOBAL_CONFIGS,
    "stage03_max_feature_size": STAGE03_MAX_FEATURE_SIZE,
    "min_inner_fold_count_round2": MIN_INNER_FOLD_COUNT_ROUND2,
    "run_outer_refit": RUN_OUTER_REFIT,
    "test_top_n": TEST_TOP_N,
    "test_row_count": TEST_ROW_COUNT,
    "test_contact_rate": TEST_CONTACT_RATE,
    "use_rate_matched_outer_selection": USE_RATE_MATCHED_OUTER_SELECTION,
    "cost_aware_max_feature_size": COST_AWARE_MAX_FEATURE_SIZE,
    "cost_aware_fallback_min_outer_folds": COST_AWARE_FALLBACK_MIN_OUTER_FOLDS,
    "top_stable_configs": TOP_STABLE_CONFIGS,
    "min_outer_folds_for_stability": MIN_OUTER_FOLDS_FOR_STABILITY,
    "scoring_formula": "10*TP - 5*FP - 200*n_features",
    "selection_contract": "Uses notebook-03 alternative outer predictions, rescored at final test contact rate.",
}
with (outputs / "stage_config.json").open("w") as file_obj:
    json.dump(config, file_obj, indent=2, default=str)

pd.Series(config)

random_state                                                                          42
max_targets                                                                         1000
outer_folds_to_run                                                                  None
stage03_global_configs                                                                20
stage03_max_feature_size                                                               6
min_inner_fold_count_round2                                                            5
run_outer_refit                                                                     True
test_top_n                                                                          1000
test_row_count                                                                      5000
test_contact_rate                                                                    0.2
use_rate_matched_outer_selection                                                    True
cost_aware_max_featur

In [3]:
OUTPUT_FILES = {
    "run_log": outputs / "outer_evaluation_run_log.csv",
    "stage02_selected_used": outputs / "stage02_selected_pipeline_recipes_used.csv",
    "outer_scores": outputs / "outer_fold_scores.csv",
    "outer_predictions": outputs / "outer_fold_predictions.csv",
    "outer_selected_features": outputs / "outer_fold_selected_features.csv",
    "outer_scores_rate_matched": outputs / "outer_fold_scores_rate_matched.csv",
    "outer_config_summary_full_cap": outputs / "outer_config_summary_full_cap.csv",
    "outer_config_summary": outputs / "outer_config_summary.csv",
    "stable_configs_full_cap": outputs / "stable_outer_configurations_full_cap.csv",
    "stable_configs": outputs / "stable_outer_configurations.csv",
    "outer_oof_scores": outputs / "outer_oof_scores.csv",
    "feature_frequency": outputs / "outer_feature_frequency.csv",
    "manifest": outputs / "output_manifest.csv",
    "status": outputs / "stage_status_summary.json",
}


def log_event(stage: str, message: str, **payload) -> None:
    row = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "stage": stage,
        "message": message,
        **payload,
    }
    frame = pd.DataFrame([row])
    frame.to_csv(
        OUTPUT_FILES["run_log"],
        mode="a",
        index=False,
        header=not OUTPUT_FILES["run_log"].exists(),
    )
    print(f"[{row['timestamp_utc']}] {stage}: {message} {payload}")


log_event("setup", "initialized notebook 03", output_dir=str(outputs.relative_to(project_root)))

[2026-06-08T14:44:58.630295+00:00] setup: initialized notebook 03 {'output_dir': 'outputs/feature_selection_alternative/03_fold_evaluation'}


## 1. Build global notebook-03 candidate configs from notebook-02 scores

The input is `round2_inner_oof_scores.csv` from notebook 02. Notebook 03 selects global `recipe_config_id` values once, then expands each selected config across all outer folds. This is intentionally different from consuming `outer_selected_pipeline_recipes.csv`, because that file is fold-local and can create survivorship bias.


In [4]:
outer_folds = sorted(outer_assignments["outer_fold"].drop_duplicates().astype(int).tolist())
if OUTER_FOLDS_TO_RUN is not None:
    outer_folds = [fold for fold in outer_folds if fold in set(OUTER_FOLDS_TO_RUN)]

REQUIRED_OUTER_FOLD_COUNT = len(outer_folds)
MIN_OUTER_FOLDS_FOR_STABILITY = REQUIRED_OUTER_FOLD_COUNT
COST_AWARE_FALLBACK_MIN_OUTER_FOLDS = REQUIRED_OUTER_FOLD_COUNT
config["required_outer_fold_count"] = REQUIRED_OUTER_FOLD_COUNT
config["min_outer_folds_for_stability"] = MIN_OUTER_FOLDS_FOR_STABILITY
config["cost_aware_fallback_min_outer_folds"] = COST_AWARE_FALLBACK_MIN_OUTER_FOLDS
with (outputs / "stage_config.json").open("w", encoding="utf-8") as handle:
    json.dump(config, handle, indent=2)

selected_pipeline_recipes = select_global_stage02_pipeline_recipes(
    stage_two_dir,
    outer_folds=outer_folds,
    top_n=STAGE03_GLOBAL_CONFIGS,
    min_inner_fold_count=MIN_INNER_FOLD_COUNT_ROUND2,
    max_feature_size=STAGE03_MAX_FEATURE_SIZE,
)
selected_pipeline_recipes.to_csv(OUTPUT_FILES["stage02_selected_used"], index=False)

log_event(
    "stage02_selection",
    "built global notebook-03 candidate configs from notebook-02 round2 scores",
    rows=len(selected_pipeline_recipes),
    global_config_count=int(selected_pipeline_recipes["recipe_config_id"].nunique())
    if not selected_pipeline_recipes.empty
    else 0,
    required_outer_fold_count=REQUIRED_OUTER_FOLD_COUNT,
    max_feature_size=STAGE03_MAX_FEATURE_SIZE,
    outer_folds=outer_folds,
)
selected_pipeline_recipes.head(30)

[2026-06-08T14:44:59.157669+00:00] stage02_selection: built global notebook-03 candidate configs from notebook-02 round2 scores {'rows': 100, 'global_config_count': 20, 'required_outer_fold_count': 5, 'max_feature_size': 6, 'outer_folds': [1, 2, 3, 4, 5]}


,recipe_config_id,pipeline_recipe_id,feature_recipe_id,prescreen_recipe_id,prescreen_name,prescreen_methods,rank_aggregation,feature_size,model_spec_id,base_model_family,...,median_inner_oof_business_score,min_inner_oof_business_score,max_inner_oof_business_score,inner_oof_optimal_k,inner_oof_f1_score,inner_oof_roc_auc_score,global_candidate_rank,inner_selection_rank,selection_artifact,outer_fold
0,ps_mean_0032__k_005__extra_trees_small_deep02,ps_mean_0032__k_005__extra_trees_small_deep02,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep02,extra_trees_small,...,4985.0,4985.0,4985.0,999.0,0.489796,0.682412,1,1,round2_inner_oof_scores.csv,1
1,ps_mean_0007__k_006__extra_trees_small_deep01,ps_mean_0007__k_006__extra_trees_small_deep01,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep01,extra_trees_small,...,5000.0,4860.0,5030.0,999.0,0.498118,0.691210,2,2,round2_inner_oof_scores.csv,1
2,ps_mean_0007__k_006__extra_trees_small_deep02,ps_mean_0007__k_006__extra_trees_small_deep02,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep02,extra_trees_small,...,4960.0,4875.0,5070.0,999.5,0.497867,0.693830,3,3,round2_inner_oof_scores.csv,1
3,ps_mean_0052__k_006__extra_trees_small_deep01,ps_mean_0052__k_006__extra_trees_small_deep01,ps_mean_0052__k_006,ps_mean_0052,triple__mutual_info+lightgbm_gain+ebm,"[""mutual_info"", ""lightgbm_gain"", ""ebm""]",mean_rank_score,6,extra_trees_small_deep01,extra_trees_small,...,4960.0,4960.0,4960.0,1000.0,0.497492,0.678581,4,4,round2_inner_oof_scores.csv,1
4,ps_mean_0032__k_005__extra_trees_small_deep03,ps_mean_0032__k_005__extra_trees_small_deep03,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep03,extra_trees_small,...,4950.0,4950.0,4950.0,1000.0,0.488294,0.683566,5,5,round2_inner_oof_scores.csv,1
5,ps_mean_0052__k_006__xgboost_classifier_deep04,ps_mean_0052__k_006__xgboost_classifier_deep04,ps_mean_0052__k_006,ps_mean_0052,triple__mutual_info+lightgbm_gain+ebm,"[""mutual_info"", ""lightgbm_gain"", ""ebm""]",mean_rank_score,6,xgboost_classifier_deep04,xgboost_classifier,...,4950.0,4950.0,4950.0,996.0,0.496824,0.688411,6,6,round2_inner_oof_scores.csv,1
6,ps_mean_0007__k_006__extra_trees_small_deep03,ps_mean_0007__k_006__extra_trees_small_deep03,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep03,extra_trees_small,...,4932.5,4890.0,5020.0,996.5,0.496823,0.694644,7,7,round2_inner_oof_scores.csv,1
7,ps_mean_0007__k_006__extra_trees_small_deep08,ps_mean_0007__k_006__extra_trees_small_deep08,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep08,extra_trees_small,...,5000.0,4750.0,5010.0,999.5,0.496697,0.693616,8,8,round2_inner_oof_scores.csv,1
8,ps_mean_0032__k_005__extra_trees_small_deep05,ps_mean_0032__k_005__extra_trees_small_deep05,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep05,extra_trees_small,...,4935.0,4935.0,4935.0,1000.0,0.487625,0.682358,9,9,round2_inner_oof_scores.csv,1
9,ps_mean_0005__k_006__extra_trees_small_deep03,ps_mean_0005__k_006__extra_trees_small_deep03,ps_mean_0005__k_006,ps_mean_0005,single__lightgbm_gain,"[""lightgbm_gain""]",mean_rank_score,6,extra_trees_small_deep03,extra_trees_small,...,4960.0,4765.0,5025.0,996.6,0.496286,0.693798,10,10,round2_inner_oof_scores.csv,1


## 2. Outer-fold evaluation (nested CV)

Each selected global config must have one completed audit row for every outer fold. Existing complete `(outer_fold, recipe_config_id)` pairs from `outer_fold_scores.csv` and `outer_fold_predictions.csv` are reused. Missing pairs are refitted from scratch on that fold's `outer_train` and scored on untouched `outer_val`.

In [5]:
def pair_set(frame: pd.DataFrame) -> set[tuple[int, str]]:
    if frame.empty or not {"outer_fold", "recipe_config_id"}.issubset(frame.columns):
        return set()
    return {
        (int(row.outer_fold), str(row.recipe_config_id))
        for row in frame[["outer_fold", "recipe_config_id"]]
        .drop_duplicates()
        .itertuples(index=False)
    }


target_pairs = pair_set(selected_pipeline_recipes)
existing_outer_scores = read_csv_if_available(OUTPUT_FILES["outer_scores"])
existing_outer_predictions = read_csv_if_available(OUTPUT_FILES["outer_predictions"])
existing_outer_selected_features = read_csv_if_available(OUTPUT_FILES["outer_selected_features"])
complete_existing_pairs = complete_outer_prediction_pairs(
    existing_outer_scores,
    existing_outer_predictions,
    outer_assignments,
    outer_folds=outer_folds,
)
reusable_pairs = target_pairs.intersection(complete_existing_pairs)
missing_pairs = target_pairs.difference(reusable_pairs)

selected_missing_recipes = selected_pipeline_recipes.loc[
    selected_pipeline_recipes.apply(
        lambda row: (int(row["outer_fold"]), str(row["recipe_config_id"])) in missing_pairs,
        axis=1,
    )
].copy()

log_event(
    "outer_refit_plan",
    "planned incremental complete outer audit",
    target_pairs=len(target_pairs),
    reusable_pairs=len(reusable_pairs),
    missing_pairs=len(missing_pairs),
    selected_missing_rows=len(selected_missing_recipes),
)

if RUN_OUTER_REFIT and not selected_missing_recipes.empty:
    new_outer_scores, new_outer_predictions, new_outer_selected_features = (
        run_outer_fold_evaluation(
            X_train,
            y_train,
            selected_missing_recipes,
            outer_assignments,
            prescreen_recipes,
            model_specs,
            outer_folds=outer_folds,
            random_state=RANDOM_STATE,
            max_targets=MAX_TARGETS,
        )
    )
else:
    new_outer_scores = pd.DataFrame()
    new_outer_predictions = pd.DataFrame()
    new_outer_selected_features = pd.DataFrame()
    if selected_missing_recipes.empty:
        log_event("outer_refit_skip", "all target pairs are already complete")
    elif not RUN_OUTER_REFIT:
        raise RuntimeError(
            "RUN_OUTER_REFIT=False but selected global configs have missing outer-fold audits. "
            "Set RUN_OUTER_REFIT=True to fit missing pairs."
        )

kept_existing_scores = filter_outer_recipe_pairs(existing_outer_scores, reusable_pairs)
kept_existing_predictions = filter_outer_recipe_pairs(existing_outer_predictions, reusable_pairs)
kept_existing_features = filter_outer_recipe_pairs(existing_outer_selected_features, reusable_pairs)

outer_fold_scores = pd.concat([kept_existing_scores, new_outer_scores], ignore_index=True)
outer_fold_predictions = pd.concat(
    [kept_existing_predictions, new_outer_predictions], ignore_index=True
)
outer_fold_selected_features = pd.concat(
    [kept_existing_features, new_outer_selected_features],
    ignore_index=True,
)

complete_after_refit = complete_outer_prediction_pairs(
    outer_fold_scores,
    outer_fold_predictions,
    outer_assignments,
    outer_folds=outer_folds,
)
still_missing_pairs = target_pairs.difference(complete_after_refit)
if still_missing_pairs:
    examples = sorted(still_missing_pairs)[:10]
    raise RuntimeError(f"Incomplete outer audit remains after refit; examples={examples}")

outer_fold_scores.to_csv(OUTPUT_FILES["outer_scores"], index=False)
outer_fold_predictions.to_csv(OUTPUT_FILES["outer_predictions"], index=False)
outer_fold_selected_features.to_csv(OUTPUT_FILES["outer_selected_features"], index=False)

log_event(
    "outer_refit",
    "saved complete alternative outer-fold evaluation frames",
    run_outer_refit=RUN_OUTER_REFIT,
    score_rows=len(outer_fold_scores),
    prediction_rows=len(outer_fold_predictions),
    selected_feature_rows=len(outer_fold_selected_features),
    complete_pairs=len(complete_after_refit),
    required_pairs=len(target_pairs),
    ok_scores=int(outer_fold_scores["status"].eq("ok").sum()) if not outer_fold_scores.empty else 0,
)
outer_fold_scores.tail(50)

[2026-06-08T14:45:06.892485+00:00] outer_refit_plan: planned incremental complete outer audit {'target_pairs': 100, 'reusable_pairs': 60, 'missing_pairs': 40, 'selected_missing_rows': 40}
[outer=1] fitting 4 prescreen methods on 4000 outer-train rows


2026-06-08 14:45:32,740 - interpret.utils._native - INFO - EBM lib loading.
2026-06-08 14:45:32,743 - interpret.utils._native - INFO - Finding library for Linux, x86_64, bitsize=64, debug=False
2026-06-08 14:45:32,777 - interpret.utils._native - INFO - Loading EBM library /workspaces/cost-effective/.venv/lib/python3.12/site-packages/interpret/utils/../root/bld/lib/libebm_linux_x64.so
2026-06-08 14:45:38,800 - interpret.utils._compressed_dataset - INFO - Creating native dataset
2026-06-08 14:45:38,940 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 14:45:38,941 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 14:45:38,967 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 14:45:45,317 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 14:45:45,320 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 14:45:45,322 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 14:45:4

[outer=1] ps_mean_0052__k_006__xgboost_classifier_deep03 score=1505.0 k=707
[outer=1] ps_mean_0052__k_006__extra_trees_small_deep02 score=1460.0 k=761
[outer=1] ps_mean_0005__k_006__extra_trees_small_deep04 score=1400.0 k=881
[outer=1] ps_mean_0032__k_005__extra_trees_small_deep06 score=1530.0 k=982
[outer=1] ps_mean_0052__k_006__xgboost_classifier_deep06 score=1485.0 k=759
[outer=1] ps_mean_0052__k_006__extra_trees_small_deep08 score=1465.0 k=805
[outer=1] ps_mean_0032__k_005__extra_trees_small_deep08 score=1545.0 k=982
[outer=1] ps_mean_0032__k_005__extra_trees_small_deep01 score=1510.0 k=980
[outer=2] fitting 4 prescreen methods on 4000 outer-train rows


2026-06-08 14:47:08,584 - interpret.utils._compressed_dataset - INFO - Creating native dataset
2026-06-08 14:47:08,731 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 14:47:08,733 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 14:47:08,763 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 14:47:13,791 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 14:47:13,792 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 14:47:13,793 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 14:47:13,793 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 14:47:13,812 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 14:47:16,841 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 14:47:16,842 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 14:47:16,843 - interpret.glassbox._ebm._boost - INFO - S

[outer=2] ps_mean_0052__k_006__xgboost_classifier_deep03 score=1345.0 k=691
[outer=2] ps_mean_0052__k_006__extra_trees_small_deep02 score=1370.0 k=752
[outer=2] ps_mean_0005__k_006__extra_trees_small_deep04 score=1470.0 k=753
[outer=2] ps_mean_0032__k_005__extra_trees_small_deep06 score=1575.0 k=919
[outer=2] ps_mean_0052__k_006__xgboost_classifier_deep06 score=1355.0 k=713
[outer=2] ps_mean_0052__k_006__extra_trees_small_deep08 score=1400.0 k=704
[outer=2] ps_mean_0032__k_005__extra_trees_small_deep08 score=1545.0 k=922
[outer=2] ps_mean_0032__k_005__extra_trees_small_deep01 score=1575.0 k=937
[outer=3] fitting 4 prescreen methods on 4000 outer-train rows


...

2026-06-08 14:49:39,726 - interpret.utils._compressed_dataset - INFO - Creating native dataset
2026-06-08 14:49:39,844 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 14:49:39,844 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 14:49:39,862 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 14:49:44,912 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 14:49:44,913 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 14:49:44,914 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 14:49:44,915 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 14:49:44,939 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 14:49:48,459 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 14:49:48,460 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 14:49:48,461 - interpret.glassbox._ebm._boost - INFO - S

[outer=4] ps_mean_0052__k_006__xgboost_classifier_deep03 score=1275.0 k=987
[outer=4] ps_mean_0052__k_006__extra_trees_small_deep02 score=1300.0 k=964
[outer=4] ps_mean_0005__k_006__extra_trees_small_deep04 score=1545.0 k=813
[outer=4] ps_mean_0032__k_005__extra_trees_small_deep06 score=1685.0 k=843
[outer=4] ps_mean_0052__k_006__xgboost_classifier_deep06 score=1280.0 k=860
[outer=4] ps_mean_0052__k_006__extra_trees_small_deep08 score=1280.0 k=971
[outer=4] ps_mean_0032__k_005__extra_trees_small_deep08 score=1700.0 k=780
[outer=4] ps_mean_0032__k_005__extra_trees_small_deep01 score=1670.0 k=831
[outer=5] fitting 4 prescreen methods on 4000 outer-train rows


2026-06-08 14:50:53,157 - interpret.utils._compressed_dataset - INFO - Creating native dataset
2026-06-08 14:50:53,275 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 14:50:53,276 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 14:50:53,293 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 14:50:56,527 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 14:50:56,529 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 14:50:56,530 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 14:50:56,531 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 14:50:56,562 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 14:51:00,033 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 14:51:00,035 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 14:51:00,035 - interpret.glassbox._ebm._boost - INFO - S

[outer=5] ps_mean_0052__k_006__xgboost_classifier_deep03 score=1425.0 k=882
[outer=5] ps_mean_0052__k_006__extra_trees_small_deep02 score=1415.0 k=851
[outer=5] ps_mean_0005__k_006__extra_trees_small_deep04 score=1525.0 k=727
[outer=5] ps_mean_0032__k_005__extra_trees_small_deep06 score=1605.0 k=595
[outer=5] ps_mean_0052__k_006__xgboost_classifier_deep06 score=1390.0 k=883
[outer=5] ps_mean_0052__k_006__extra_trees_small_deep08 score=1425.0 k=813
[outer=5] ps_mean_0032__k_005__extra_trees_small_deep08 score=1615.0 k=896
[outer=5] ps_mean_0032__k_005__extra_trees_small_deep01 score=1625.0 k=648
[2026-06-08T14:52:01.453387+00:00] outer_refit: saved complete alternative outer-fold evaluation frames {'run_outer_refit': True, 'score_rows': 100, 'prediction_rows': 100000, 'selected_feature_rows': 570, 'complete_pairs': 100, 'required_pairs': 100, 'ok_scores': 100}


,recipe_config_id,pipeline_recipe_id,feature_recipe_id,prescreen_recipe_id,prescreen_name,prescreen_methods,rank_aggregation,feature_size,model_spec_id,base_model_family,...,outer_optimal_k,outer_tp_at_optimal_k,outer_fp_at_optimal_k,outer_gross_score,feature_penalty,outer_f1_score,outer_roc_auc_score,outer_n_predictions,outer_selected_rate,outer_threshold
50,ps_mean_0032__k_005__extra_trees_small_deep05,ps_mean_0032__k_005__extra_trees_small_deep05,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep05,extra_trees_small,...,619,382,237,2635.0,1000.0,0.683975,0.698675,1000,0.619,0.476226
51,ps_mean_0007__k_006__extra_trees_small_deep03,ps_mean_0007__k_006__extra_trees_small_deep03,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep03,extra_trees_small,...,731,428,303,2765.0,1200.0,0.696763,0.712115,1000,0.731,0.474153
52,ps_mean_0007__k_006__extra_trees_small_deep02,ps_mean_0007__k_006__extra_trees_small_deep02,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep02,extra_trees_small,...,690,414,276,2760.0,1200.0,0.696970,0.710995,1000,0.690,0.479743
53,ps_mean_0007__k_006__extra_trees_small_deep08,ps_mean_0007__k_006__extra_trees_small_deep08,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep08,extra_trees_small,...,735,429,306,2760.0,1200.0,0.695864,0.710175,1000,0.735,0.472345
54,ps_mean_0007__k_006__extra_trees_small_deep04,ps_mean_0007__k_006__extra_trees_small_deep04,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep04,extra_trees_small,...,703,418,285,2755.0,1200.0,0.696087,0.707883,1000,0.703,0.474570
55,ps_mean_0007__k_006__extra_trees_small_deep01,ps_mean_0007__k_006__extra_trees_small_deep01,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep01,extra_trees_small,...,709,419,290,2740.0,1200.0,0.694283,0.708415,1000,0.709,0.483583
56,ps_mean_0005__k_006__extra_trees_small_deep03,ps_mean_0005__k_006__extra_trees_small_deep03,ps_mean_0005__k_006,ps_mean_0005,single__lightgbm_gain,"[""lightgbm_gain""]",mean_rank_score,6,extra_trees_small_deep03,extra_trees_small,...,870,472,398,2730.0,1200.0,0.690058,0.698771,1000,0.870,0.466574
57,ps_mean_0005__k_006__extra_trees_small_deep02,ps_mean_0005__k_006__extra_trees_small_deep02,ps_mean_0005__k_006,ps_mean_0005,single__lightgbm_gain,"[""lightgbm_gain""]",mean_rank_score,6,extra_trees_small_deep02,extra_trees_small,...,864,470,394,2730.0,1200.0,0.690162,0.698959,1000,0.864,0.472349
58,ps_mean_0052__k_006__extra_trees_small_deep01,ps_mean_0052__k_006__extra_trees_small_deep01,ps_mean_0052__k_006,ps_mean_0052,triple__mutual_info+lightgbm_gain+ebm,"[""mutual_info"", ""lightgbm_gain"", ""ebm""]",mean_rank_score,6,extra_trees_small_deep01,extra_trees_small,...,766,431,335,2635.0,1200.0,0.681962,0.693127,1000,0.766,0.481348
59,ps_mean_0052__k_006__xgboost_classifier_deep04,ps_mean_0052__k_006__xgboost_classifier_deep04,ps_mean_0052__k_006,ps_mean_0052,triple__mutual_info+lightgbm_gain+ebm,"[""mutual_info"", ""lightgbm_gain"", ""ebm""]",mean_rank_score,6,xgboost_classifier_deep04,xgboost_classifier,...,810,443,367,2595.0,1200.0,0.677370,0.685081,1000,0.810,0.385782


## 3. Rate-matched, population-scaled finalist selection

The raw outer score allows `k <= 1000` on a 1000-row outer fold, which effectively permits contacting the entire fold. The final test ranking selects 1000 from about 5000 rows, so finalist selection below caps each outer fold at the same contact rate, around 200 rows.

Because one outer fold is only about one fifth of the final test population, the TP/FP gross value is scaled back to the full test row count before subtracting feature cost once. The main `stable_outer_configurations.csv` is selected from this scaled score, with a cost-aware feature-size preference. The old full-cap summary is still saved for diagnostics.


In [6]:
outer_val_row_counts = (
    outer_assignments.loc[outer_assignments["outer_fold"].isin(outer_folds)]
    .groupby("outer_fold")["sample_index"]
    .size()
    .to_dict()
)
rate_matched_max_targets_by_outer = {
    int(outer_fold): max(1, min(int(row_count), round(row_count * TEST_CONTACT_RATE)))
    for outer_fold, row_count in outer_val_row_counts.items()
}
gross_score_scale_by_outer = {
    int(outer_fold): float(TEST_ROW_COUNT) / float(row_count)
    for outer_fold, row_count in outer_val_row_counts.items()
}

outer_fold_scores_rate_matched = rescore_outer_predictions_at_targets(
    outer_fold_predictions,
    max_targets_by_outer_fold=rate_matched_max_targets_by_outer,
    gross_score_scale_by_outer_fold=gross_score_scale_by_outer,
    original_scores=outer_fold_scores,
    scoring_strategy="rate_matched_test_top1000_contact_rate_scaled_gross",
)
outer_fold_scores_rate_matched.to_csv(OUTPUT_FILES["outer_scores_rate_matched"], index=False)

outer_config_summary_full_cap = aggregate_outer_scores(outer_fold_scores)
outer_config_summary = aggregate_outer_scores(outer_fold_scores_rate_matched)

stable_outer_configurations_full_cap = select_stable_configurations(
    outer_config_summary_full_cap.loc[
        outer_config_summary_full_cap["outer_fold_count"].eq(REQUIRED_OUTER_FOLD_COUNT)
    ].copy(),
    top_n=TOP_STABLE_CONFIGS,
    min_outer_folds=REQUIRED_OUTER_FOLD_COUNT,
)
stable_outer_configurations = select_cost_aware_configurations(
    outer_config_summary,
    top_n=TOP_STABLE_CONFIGS,
    min_outer_folds=REQUIRED_OUTER_FOLD_COUNT,
    max_feature_size=COST_AWARE_MAX_FEATURE_SIZE,
    fallback_min_outer_folds=REQUIRED_OUTER_FOLD_COUNT,
    require_exact_outer_folds=True,
    allow_incomplete_fallback=False,
)
if (
    not stable_outer_configurations.empty
    and not stable_outer_configurations["outer_fold_count"].eq(REQUIRED_OUTER_FOLD_COUNT).all()
):
    raise RuntimeError("Final stable configs include incomplete outer-fold audits")

outer_config_summary_full_cap.to_csv(OUTPUT_FILES["outer_config_summary_full_cap"], index=False)
outer_config_summary.to_csv(OUTPUT_FILES["outer_config_summary"], index=False)
stable_outer_configurations_full_cap.to_csv(OUTPUT_FILES["stable_configs_full_cap"], index=False)
stable_outer_configurations.to_csv(OUTPUT_FILES["stable_configs"], index=False)

log_event(
    "aggregation",
    "saved rate-matched population-scaled cost-aware finalists",
    contact_rate=TEST_CONTACT_RATE,
    max_targets_by_outer=rate_matched_max_targets_by_outer,
    gross_score_scale_by_outer=gross_score_scale_by_outer,
    full_cap_config_rows=len(outer_config_summary_full_cap),
    rate_matched_config_rows=len(outer_config_summary),
    finalist_rows=len(stable_outer_configurations),
    required_outer_fold_count=REQUIRED_OUTER_FOLD_COUNT,
    cost_aware_max_feature_size=COST_AWARE_MAX_FEATURE_SIZE,
)
stable_outer_configurations

[2026-06-08T14:53:29.775931+00:00] aggregation: saved rate-matched population-scaled cost-aware finalists {'contact_rate': 0.2, 'max_targets_by_outer': {1: 200, 2: 200, 3: 200, 4: 200, 5: 200}, 'gross_score_scale_by_outer': {1: 5.0, 2: 5.0, 3: 5.0, 4: 5.0, 5: 5.0}, 'full_cap_config_rows': 20, 'rate_matched_config_rows': 20, 'finalist_rows': 10, 'required_outer_fold_count': 5, 'cost_aware_max_feature_size': 6}


,recipe_config_id,prescreen_recipe_id,prescreen_name,feature_size,model_spec_id,base_model_family,model_kind,model_params,outer_fold_count,mean_outer_business_score,...,mean_outer_f1_score,mean_outer_roc_auc_score,mean_inner_oof_business_score,mean_inner_selection_rank,selected_feature_count,example_selected_features_text,outer_config_rank,selection_note,stability_note,finalist_rank
0,ps_mean_0007__k_006__extra_trees_small_deep03,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep03,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 5, ""...",5,5450.0,...,0.445211,0.695838,4943.75,7.0,6.0,"var_175,var_214,var_254,var_341,var_379,var_482",1,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,1
1,ps_mean_0007__k_006__extra_trees_small_deep08,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep08,extra_trees_small,classifier,"{""class_weight"": null, ""max_depth"": 5, ""min_sa...",5,5400.0,...,0.443361,0.695104,4940.00,8.0,6.0,"var_175,var_214,var_254,var_341,var_379,var_482",2,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,2
2,ps_mean_0007__k_006__extra_trees_small_deep04,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep04,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 6, ""...",5,5355.0,...,0.441444,0.694526,4931.25,11.0,6.0,"var_175,var_214,var_254,var_341,var_379,var_482",3,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,3
3,ps_mean_0007__k_006__extra_trees_small_deep02,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep02,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 4, ""...",5,5320.0,...,0.440046,0.693882,4966.25,3.0,6.0,"var_175,var_214,var_254,var_341,var_379,var_482",4,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,4
4,ps_mean_0007__k_006__extra_trees_small_deep01,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep01,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 3, ""...",5,5310.0,...,0.439727,0.693065,4972.50,2.0,6.0,"var_175,var_214,var_254,var_341,var_379,var_482",5,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,5
5,ps_mean_0005__k_006__extra_trees_small_deep02,ps_mean_0005,single__lightgbm_gain,6,extra_trees_small_deep02,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 4, ""...",5,5270.0,...,0.438327,0.692887,4930.00,12.0,6.0,"var_159,var_190,var_214,var_254,var_341,var_379",6,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,6
6,ps_mean_0005__k_006__extra_trees_small_deep04,ps_mean_0005,single__lightgbm_gain,6,extra_trees_small_deep04,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 6, ""...",5,5255.0,...,0.437757,0.693693,4923.00,15.0,6.0,"var_159,var_190,var_214,var_254,var_341,var_379",7,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,7
7,ps_mean_0032__k_005__extra_trees_small_deep05,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,5,extra_trees_small_deep05,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 7, ""...",5,5205.0,...,0.428323,0.678304,4935.00,9.0,5.0,"var_175,var_214,var_254,var_341,var_379",8,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,8
8,ps_mean_0032__k_005__extra_trees_small_deep01,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,5,extra_trees_small_deep01,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 3, ""...",5,5160.0,...,0.426601,0.675378,4905.00,20.0,5.0,"var_175,var_214,var_254,var_341,var_379",9,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,9
9,ps_mean_0005__k_006__extra_trees_small_deep03,ps_mean_0005,single__lightgbm_gain,6,extra_trees_small_deep03,extra_trees_small,classifier,"{""class_weight"": ""

In [7]:
if not outer_fold_selected_features.empty:
    outer_feature_frequency = (
        outer_fold_selected_features.loc[outer_fold_selected_features["status"].eq("ok")]
        .groupby("feature", as_index=False)
        .agg(
            count=("feature", "size"),
            outer_fold_count=("outer_fold", "nunique"),
            mean_feature_order=("feature_order", "mean"),
            recipe_config_count=("recipe_config_id", "nunique"),
        )
        .sort_values(
            ["count", "outer_fold_count", "mean_feature_order"], ascending=[False, False, True]
        )
        .reset_index(drop=True)
    )
else:
    outer_feature_frequency = pd.DataFrame()

outer_feature_frequency.to_csv(OUTPUT_FILES["feature_frequency"], index=False)
outer_feature_frequency.head(50)

,feature,count,outer_fold_count,mean_feature_order,recipe_config_count
0,var_254,89,5,3.561798,20
1,var_379,88,5,5.306818,20
2,var_175,73,5,1.123288,20
3,var_341,71,5,4.464789,20
4,var_190,69,4,2.130435,20
5,var_214,68,4,2.632353,20
6,var_159,33,4,1.181818,9
7,var_482,25,5,6.000000,5
8,var_264,12,2,4.500000,6
9,var_463,12,2,6.000000,6


## 3b. Pooled outer OOF score

Each config produced predictions on 5 disjoint outer_val folds (1000 rows each). Concatenating them gives 5000 predictions that together cover the full X_train exactly once — a standard OOF setup. Scoring at `max_targets=1000` corresponds directly to the real deployment contact rate (1000/5000 = 20%), with no population scaling or per-fold averaging needed.

This is the outer-fold analogue of the Stage 2 inner-OOF pooling and serves as a direct, unscaled additional metric alongside the rate-matched per-fold scores.

In [8]:
import numpy as np

outer_oof_rows = []
ok_preds = outer_fold_predictions[outer_fold_predictions["status"] == "ok"]

for config_id, group in ok_preds.groupby("recipe_config_id"):
    feature_size = int(group["feature_size"].iloc[0])
    n_folds = int(group["outer_fold"].nunique())
    n_samples = len(group)

    if group["sample_index"].duplicated().any():
        outer_oof_rows.append({
            "recipe_config_id": config_id,
            "outer_oof_n_predictions": n_samples,
            "outer_oof_n_folds": n_folds,
            "outer_oof_business_score": None,
            "outer_oof_optimal_k": None,
            "status": "duplicate_samples",
        })
        continue

    y_arr = group["y_true"].values.astype(int)
    s_arr = group["score"].values.astype(float)

    target_limit = min(MAX_TARGETS, n_samples)
    feature_penalty = 200 * feature_size

    order = np.argsort(s_arr)[::-1]
    ranked_y = y_arr[order]

    cumtp = np.cumsum(ranked_y[:target_limit] == 1)
    cumfp = np.cumsum(ranked_y[:target_limit] == 0)
    business_at_k = 10 * cumtp - 5 * cumfp - feature_penalty

    best_k = int(np.argmax(business_at_k)) + 1
    best_score = float(business_at_k[best_k - 1])

    outer_oof_rows.append({
        "recipe_config_id": config_id,
        "outer_oof_n_predictions": n_samples,
        "outer_oof_n_folds": n_folds,
        "outer_oof_business_score": best_score,
        "outer_oof_optimal_k": best_k,
        "status": "ok",
    })

outer_oof_scores = (
    pd.DataFrame(outer_oof_rows)
    .sort_values("outer_oof_business_score", ascending=False)
    .reset_index(drop=True)
)
outer_oof_scores.to_csv(OUTPUT_FILES["outer_oof_scores"], index=False)

log_event(
    "outer_oof_pooling",
    "computed pooled outer OOF scores (5000 samples per config at 20% contact rate)",
    config_count=len(outer_oof_scores),
    ok_count=int(outer_oof_scores["status"].eq("ok").sum()),
    best_outer_oof_score=float(outer_oof_scores.iloc[0]["outer_oof_business_score"])
    if not outer_oof_scores.empty
    else None,
    best_outer_oof_config=str(outer_oof_scores.iloc[0]["recipe_config_id"])
    if not outer_oof_scores.empty
    else None,
)
outer_oof_scores

[2026-06-08T14:53:37.139935+00:00] outer_oof_pooling: computed pooled outer OOF scores (5000 samples per config at 20% contact rate) {'config_count': 20, 'ok_count': 20, 'best_outer_oof_score': 5445.0, 'best_outer_oof_config': 'ps_mean_0007__k_006__extra_trees_small_deep01'}


,recipe_config_id,outer_oof_n_predictions,outer_oof_n_folds,outer_oof_business_score,outer_oof_optimal_k,status
0,ps_mean_0007__k_006__extra_trees_small_deep01,5000,5,5445.0,999,ok
1,ps_mean_0007__k_006__extra_trees_small_deep03,5000,5,5425.0,997,ok
2,ps_mean_0007__k_006__extra_trees_small_deep04,5000,5,5410.0,1000,ok
3,ps_mean_0007__k_006__extra_trees_small_deep02,5000,5,5395.0,1000,ok
4,ps_mean_0007__k_006__extra_trees_small_deep08,5000,5,5295.0,999,ok
5,ps_mean_0005__k_006__extra_trees_small_deep02,5000,5,5260.0,1000,ok
6,ps_mean_0005__k_006__extra_trees_small_deep04,5000,5,5245.0,1000,ok
7,ps_mean_0032__k_005__extra_trees_small_deep05,5000,5,5220.0,1000,ok
8,ps_mean_0032__k_005__extra_trees_small_deep03,5000,5,5205.0,1000,ok
9,ps_mean_0032__k_005__extra_trees_small_deep01,5000,5,5190.0,1000,ok


## 4. Output manifest


In [10]:
status_summary = {
    "scoring_version": "alternative_feature_selection_rate_matched_topk_v3_scaled_gross",
    "selection_strategy": "rate_matched_contact_rate_scaled_gross_cost_aware",
    "test_top_n": TEST_TOP_N,
    "test_row_count": TEST_ROW_COUNT,
    "test_contact_rate": TEST_CONTACT_RATE,
    "rate_matched_max_targets_by_outer": rate_matched_max_targets_by_outer,
    "gross_score_scale_by_outer": gross_score_scale_by_outer,
    "cost_aware_max_feature_size": COST_AWARE_MAX_FEATURE_SIZE,
    "required_outer_fold_count": REQUIRED_OUTER_FOLD_COUNT,
    "run_outer_refit": RUN_OUTER_REFIT,
    "score_rows_full_cap": len(outer_fold_scores),
    "score_rows_rate_matched": len(outer_fold_scores_rate_matched),
    "ok_score_rows_rate_matched": int(outer_fold_scores_rate_matched["status"].eq("ok").sum())
    if not outer_fold_scores_rate_matched.empty
    else 0,
    "prediction_rows": len(outer_fold_predictions),
    "selected_feature_rows": len(outer_fold_selected_features),
    "outer_config_rows_full_cap": len(outer_config_summary_full_cap),
    "outer_config_rows_rate_matched": len(outer_config_summary),
    "stable_config_rows": len(stable_outer_configurations),
    "best_mean_outer_business_score_rate_matched": float(
        outer_config_summary.iloc[0]["mean_outer_business_score"]
    )
    if not outer_config_summary.empty
    else None,
    "best_recipe_config_id_rate_matched": str(outer_config_summary.iloc[0]["recipe_config_id"])
    if not outer_config_summary.empty
    else None,
    "best_outer_oof_business_score": float(outer_oof_scores.iloc[0]["outer_oof_business_score"])
    if not outer_oof_scores.empty and outer_oof_scores.iloc[0]["status"] == "ok"
    else None,
    "best_recipe_config_id_outer_oof": str(outer_oof_scores.iloc[0]["recipe_config_id"])
    if not outer_oof_scores.empty
    else None,
}
with OUTPUT_FILES["status"].open("w") as file_obj:
    json.dump(status_summary, file_obj, indent=2, default=str)

output_manifest = pd.DataFrame([
    {"file": "stage_config.json", "meaning": "Notebook-03 execution config."},
    {"file": "outer_evaluation_run_log.csv", "meaning": "Timestamped notebook-03 log."},
    {
        "file": "stage02_selected_pipeline_recipes_used.csv",
        "meaning": "Notebook-02 selected recipes consumed by outer evaluation.",
    },
    {
        "file": "outer_fold_scores.csv",
        "meaning": "Original full-cap outer scores on untouched outer_val.",
    },
    {
        "file": "outer_fold_predictions.csv",
        "meaning": "Sample-level outer_val predictions for audited recipes.",
    },
    {
        "file": "outer_fold_selected_features.csv",
        "meaning": "Features selected after refitting prescreening on each outer_train.",
    },
    {
        "file": "outer_fold_scores_rate_matched.csv",
        "meaning": "Outer predictions rescored at final test contact rate with gross value scaled to full test population, without refitting.",
    },
    {
        "file": "outer_config_summary_full_cap.csv",
        "meaning": "Diagnostic aggregation using the original full-cap outer scores.",
    },
    {
        "file": "outer_config_summary.csv",
        "meaning": "Main aggregation using rate-matched, scaled-gross outer scores.",
    },
    {
        "file": "stable_outer_configurations_full_cap.csv",
        "meaning": "Diagnostic finalists from the old full-cap strategy.",
    },
    {
        "file": "stable_outer_configurations.csv",
        "meaning": "Main finalists selected from rate-matched alternative outer evaluation; every row has outer_fold_count equal to the full outer-fold count.",
    },
    {
        "file": "outer_oof_scores.csv",
        "meaning": "Pooled outer OOF scores: 5x1000 disjoint outer_val predictions concatenated and scored once at max_targets=1000 (20% contact rate). Direct unscaled analogue of Stage 2 inner-OOF pooling.",
    },
    {
        "file": "outer_feature_frequency.csv",
        "meaning": "Feature frequency across alternative outer refits.",
    },
    {"file": "stage_status_summary.json", "meaning": "Compact notebook-03 status summary."},
])
output_manifest.to_csv(OUTPUT_FILES["manifest"], index=False)

pd.Series(status_summary)

scoring_version                                alternative_feature_selection_rate_matched_top...
selection_strategy                             rate_matched_contact_rate_scaled_gross_cost_aware
test_top_n                                                                                  1000
test_row_count                                                                              5000
test_contact_rate                                                                            0.2
rate_matched_max_targets_by_outer                       {1: 200, 2: 200, 3: 200, 4: 200, 5: 200}
gross_score_scale_by_outer                              {1: 5.0, 2: 5.0, 3: 5.0, 4: 5.0, 5: 5.0}
cost_aware_max_feature_size                                                                    6
required_outer_fold_count                                                                      5
run_outer_refit                                                                             True
score_rows_full_cap           